
# 03 — Build the Examination Conflict Graph

## Objective

Construct an undirected graph representing examination conflicts
using the validated University of Nottingham benchmark dataset.

- Each vertex represents one examination.
- An edge connects two examinations if at least one student
  is enrolled in both.
- Duplicate student–exam relationships are excluded.
- Graph properties and structural integrity will be evaluated.
- The resulting graph will be saved for baseline algorithm testing.

The graph will serve as the input to the graph-colouring algorithms
in the next stage of the research.


In [1]:

from pathlib import Path
from collections import defaultdict
from itertools import combinations
import pickle

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Locate the project root
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data" / "raw" / "Nott").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate data/raw/Nott")
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "Nott"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory exists:", DATA_DIR.exists())
print("Processed directory:", PROCESSED_DIR)
print("NetworkX version:", nx.__version__)


Project root: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling
Raw data directory exists: True
Processed directory: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed
NetworkX version: 3.6.1


In [2]:

# Load exam IDs from the fixed-width exam file
with open(DATA_DIR / "exams", "r", encoding="utf-8", errors="replace") as f:
    exam_ids = {
        line[:8].strip()
        for line in f
        if line.strip()
    }

# Read raw enrolment pairs
raw_enrolment_pairs = []

with open(DATA_DIR / "enrolements", "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.rstrip("\r\n")

        if len(line) >= 18:
            student_id = line[:10].strip()
            original_exam_id = line[10:18].strip()

            if student_id and original_exam_id:
                raw_enrolment_pairs.append((student_id, original_exam_id))

# Normalize enrolment exam IDs
special_exam_mapping = {"K1AHWAE": "K1AHWAE2"}

normalized_pairs = []
unmatched_exam_ids = set()

for student_id, original_exam_id in raw_enrolment_pairs:
    if original_exam_id in exam_ids:
        normalized_exam_id = original_exam_id
    elif original_exam_id in special_exam_mapping:
        normalized_exam_id = special_exam_mapping[original_exam_id]
    elif original_exam_id + "1" in exam_ids:
        normalized_exam_id = original_exam_id + "1"
    else:
        unmatched_exam_ids.add(original_exam_id)
        continue

    normalized_pairs.append((student_id, normalized_exam_id))

# Remove duplicate student-exam relationships
unique_enrolment_pairs = sorted(set(normalized_pairs))

print("Unique examinations:", len(exam_ids))
print("Raw enrolment records:", len(raw_enrolment_pairs))
print("Mapped enrolment records:", len(normalized_pairs))
print("Unique student-exam relationships:", len(unique_enrolment_pairs))
print("Unmatched exam IDs:", len(unmatched_exam_ids))

assert len(exam_ids) == 800
assert len(normalized_pairs) == 33997
assert len(unique_enrolment_pairs) == 33866
assert not unmatched_exam_ids

print("Dataset preparation checks passed.")


Unique examinations: 800
Raw enrolment records: 33997
Mapped enrolment records: 33997
Unique student-exam relationships: 33866
Unmatched exam IDs: 0
Dataset preparation checks passed.


In [3]:

# Group examinations by student
student_to_exams = defaultdict(list)

for student_id, exam_id in unique_enrolment_pairs:
    student_to_exams[student_id].append(exam_id)

# Create an undirected graph with every examination as a vertex
G = nx.Graph()
G.add_nodes_from(sorted(exam_ids))

# Connect examinations taken by the same student
for exams_for_student in student_to_exams.values():
    for exam_a, exam_b in combinations(exams_for_student, 2):
        G.add_edge(exam_a, exam_b)

print("Conflict graph constructed successfully.")
print("Number of vertices:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print("Is the graph undirected:", not G.is_directed())
print("Has self-loops:", nx.number_of_selfloops(G) > 0)

assert G.number_of_nodes() == 800
assert nx.number_of_selfloops(G) == 0

print("Basic graph construction checks passed.")


Conflict graph constructed successfully.
Number of vertices: 800
Number of edges: 9990
Is the graph undirected: True
Has self-loops: False
Basic graph construction checks passed.


In [4]:
# Calculate graph structure statistics
degrees = dict(G.degree())
components = list(nx.connected_components(G))
isolated_exams = list(nx.isolates(G))

print("Graph structure statistics")
print("-" * 35)
print("Vertices:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Graph density:", round(nx.density(G), 5))
print("Connected components:", len(components))
print("Largest component:", max(len(c) for c in components))
print("Maximum degree:", max(degrees.values()))
print("Average degree:", round(sum(degrees.values()) / len(degrees), 3))
print("Isolated examinations:", sorted(isolated_exams))

print("\nComponent sizes:")
print(sorted((len(c) for c in components), reverse=True))

# Basic integrity checks
assert G.number_of_nodes() == 800
assert nx.number_of_selfloops(G) == 0
assert all(degree >= 0 for degree in degrees.values())

print("\nGraph integrity checks passed.")

Graph structure statistics
-----------------------------------
Vertices: 800
Edges: 9990
Graph density: 0.03126
Connected components: 14
Largest component: 780
Maximum degree: 203
Average degree: 24.975
Isolated examinations: ['C81J1AE2', 'C81J2AE2', 'C81J3AE2', 'C81J4AE2', 'C81J5AE2', 'C81JIAE2', 'F12X04E2', 'F12X04E3', 'H3BFM2E2', 'Q8P010E1', 'V6B335E1']

Component sizes:
[780, 6, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Graph integrity checks passed.


In [5]:
# Find examinations that have no enrolment records
enrolled_exam_ids = {
    exam_id for _, exam_id in unique_enrolment_pairs
}

exams_without_enrolments = sorted(exam_ids - enrolled_exam_ids)

print("Examinations with no enrolment records:",
      len(exams_without_enrolments))

for exam_id in exams_without_enrolments:
    print(exam_id)

# Show the number of students enrolled in each examination
exam_enrolment_counts = defaultdict(int)

for student_id, exam_id in unique_enrolment_pairs:
    exam_enrolment_counts[exam_id] += 1

print("\nExaminations with enrolments but no conflict edges:")
for exam_id in sorted(nx.isolates(G)):
    print(
        exam_id,
        "| Students enrolled:",
        exam_enrolment_counts.get(exam_id, 0)
    )

Examinations with no enrolment records: 9
C81J1AE2
C81J2AE2
C81J3AE2
C81J4AE2
C81J5AE2
C81JIAE2
F12X04E2
F12X04E3
H3BFM2E2

Examinations with enrolments but no conflict edges:
C81J1AE2 | Students enrolled: 0
C81J2AE2 | Students enrolled: 0
C81J3AE2 | Students enrolled: 0
C81J4AE2 | Students enrolled: 0
C81J5AE2 | Students enrolled: 0
C81JIAE2 | Students enrolled: 0
F12X04E2 | Students enrolled: 0
F12X04E3 | Students enrolled: 0
H3BFM2E2 | Students enrolled: 0
Q8P010E1 | Students enrolled: 1
V6B335E1 | Students enrolled: 1


In [6]:
# Inspect original exam identifiers that required normalization
from collections import Counter

original_id_counts = Counter(
    original_exam_id
    for _, original_exam_id in raw_enrolment_pairs
)

normalization_summary = {
    "Already matched": 0,
    "Special mapping": 0,
    "Added trailing 1": 0,
    "Unmatched": 0,
}

for original_exam_id in original_id_counts:
    if original_exam_id in exam_ids:
        normalization_summary["Already matched"] += 1
    elif original_exam_id in special_exam_mapping:
        normalization_summary["Special mapping"] += 1
    elif original_exam_id + "1" in exam_ids:
        normalization_summary["Added trailing 1"] += 1
    else:
        normalization_summary["Unmatched"] += 1

print("Normalization summary (distinct original IDs):")
for category, count in normalization_summary.items():
    print(f"{category}: {count}")

print("\nOriginal exam IDs that did not match any rule:")
for original_exam_id in sorted(original_id_counts):
    if (
        original_exam_id not in exam_ids
        and original_exam_id not in special_exam_mapping
        and original_exam_id + "1" not in exam_ids
    ):
        print(repr(original_exam_id), "records:", original_id_counts[original_exam_id])

Normalization summary (distinct original IDs):
Already matched: 0
Special mapping: 1
Added trailing 1: 790
Unmatched: 0

Original exam IDs that did not match any rule:


In [7]:
# Independently reconstruct the expected conflict-edge set
expected_edges = set()

for exams_for_student in student_to_exams.values():
    unique_exams = sorted(set(exams_for_student))

    for exam_a, exam_b in combinations(unique_exams, 2):
        expected_edges.add((exam_a, exam_b))

actual_edges = {
    tuple(sorted((exam_a, exam_b)))
    for exam_a, exam_b in G.edges()
}

missing_edges = expected_edges - actual_edges
unexpected_edges = actual_edges - expected_edges

print("Expected unique conflict edges:", len(expected_edges))
print("Actual graph edges:", len(actual_edges))
print("Missing edges:", len(missing_edges))
print("Unexpected edges:", len(unexpected_edges))

assert not missing_edges
assert not unexpected_edges

print("Conflict-edge validation passed.")

Expected unique conflict edges: 9990
Actual graph edges: 9990
Missing edges: 0
Unexpected edges: 0
Conflict-edge validation passed.


In [8]:
# Save the validated conflict graph
graph_path = PROCESSED_DIR / "conflict_graph.gpickle"

with open(graph_path, "wb") as f:
    pickle.dump(G, f)

print("Graph saved to:", graph_path)
print("File exists:", graph_path.exists())
print("File size (bytes):", graph_path.stat().st_size)

# Reload the graph to verify the saved file
with open(graph_path, "rb") as f:
    G_loaded = pickle.load(f)

assert G_loaded.number_of_nodes() == G.number_of_nodes()
assert G_loaded.number_of_edges() == G.number_of_edges()
assert nx.number_of_selfloops(G_loaded) == 0

print("\nSaved graph successfully reloaded.")
print("Reloaded vertices:", G_loaded.number_of_nodes())
print("Reloaded edges:", G_loaded.number_of_edges())
print("Graph persistence checks passed.")

Graph saved to: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\conflict_graph.gpickle
File exists: True
File size (bytes): 249029

Saved graph successfully reloaded.
Reloaded vertices: 800
Reloaded edges: 9990
Graph persistence checks passed.


In [9]:
# Final summary of the validated conflict graph
graph_summary = {
    "Number of examinations (vertices)": G.number_of_nodes(),
    "Number of student-conflict edges": G.number_of_edges(),
    "Graph density": round(nx.density(G), 5),
    "Connected components": nx.number_connected_components(G),
    "Isolated examinations": len(list(nx.isolates(G))),
    "Maximum examination conflicts": max(dict(G.degree()).values()),
    "Average examination conflicts": round(
        sum(dict(G.degree()).values()) / G.number_of_nodes(), 3
    ),
}

for metric, value in graph_summary.items():
    print(f"{metric}: {value}")

Number of examinations (vertices): 800
Number of student-conflict edges: 9990
Graph density: 0.03126
Connected components: 14
Isolated examinations: 11
Maximum examination conflicts: 203
Average examination conflicts: 24.975


## Conflict Graph Construction — Conclusions

The validated University of Nottingham examination dataset was
converted into an undirected conflict graph.

Each vertex represents an examination. An edge connects two
examinations when at least one student is enrolled in both.

The graph contains 800 examination vertices and 9,990 unique
conflict edges. Duplicate student–examination relationships were
removed before graph construction. The graph was checked for
self-loops, and its edge set was independently validated against
the normalized enrolment data.

The graph was saved to `data/processed/conflict_graph.gpickle`
and successfully reloaded with the expected number of vertices
and edges.

Nine examinations have no enrolment records in the supplied
enrolment data. Two additional examinations are isolated because
each has one enrolled student and no shared-student conflict.

This graph provides the input for evaluating baseline graph-colouring
algorithms. Graph colouring addresses examination conflicts, but
does not by itself enforce all calendar, room, or other operational
constraints of a complete examination timetable.